# Module 3 — Precision/Recall & Decision Trees
**Lectures:** Week 5 (precision, recall, F1, decision trees, entropy, information gain, overfitting) · Week 6 (plot_tree, max_depth, pruning)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ieso_lab import load_dataset
from ieso_lab.features import add_peak_label
from ieso_lab.tracking import log_run, show_runs

plt.rcParams["figure.figsize"] = (9, 4)
df = load_dataset().dropna().reset_index(drop=True)
print(df.shape)

In [ ]:
df = df.sort_values("timestamp_utc").reset_index(drop=True)
cut = int(len(df) * 0.8)
tr, te = df.iloc[:cut].copy(), df.iloc[cut:].copy()
thr = tr["demand_mw"].quantile(0.90)
tr, te = add_peak_label(tr, thr), add_peak_label(te, thr)
FEATURES = ["hour", "dayofweek", "month", "is_weekend", "is_holiday", "temperature_2m", "hdd", "cdd", "demand_lag_24h"]
X_train, y_train, X_test, y_test = tr[FEATURES], tr["is_peak"], te[FEATURES], te["is_peak"]

## 1. Entropy by hand
$H(S) = -\sum_c p_c \log_2 p_c$ · Information gain = $H(\text{parent}) - \sum \frac{|S_k|}{|S|} H(S_k)$

In [ ]:
def entropy(y):
    # TODO 9: return entropy of a 0/1 array (handle the case where a class is absent)
    pass

def info_gain(y, mask):
    # TODO 10: parent entropy minus weighted entropy of y[mask] and y[~mask]
    pass

print("Root entropy:", entropy(y_train.values))
for rule, mask in {"hour >= 16": X_train.hour >= 16,
                   "temp >= 25": X_train.temperature_2m >= 25,
                   "is_weekend": X_train.is_weekend == 1}.items():
    print(f"{rule:12s} gain = {info_gain(y_train.values, mask.values):.4f}")

The tree picks the split with the **largest gain** — greedily, at every node.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
# TODO 11: DecisionTreeClassifier(criterion="entropy", max_depth=4, random_state=11), fit, predict
tree = None
y_pred = None
metrics = {"accuracy": accuracy_score(y_test, y_pred), "precision": precision_score(y_test, y_pred),
           "recall": recall_score(y_test, y_pred), "f1": f1_score(y_test, y_pred)}
print(metrics)

In [ ]:
log_run("03", "tree", {"max_depth": 4}, metrics)
plt.figure(figsize=(16, 7))
plot_tree(tree, feature_names=FEATURES, class_names=["normal", "peak"], max_depth=2, filled=True, fontsize=8)

Precision = TP/(TP+FP): "when I say peak, am I right?" · Recall = TP/(TP+FN): "of all real peaks, how many did I catch?"

## 🔨 Break it: grow the tree until it memorises
**Predict:** what happens to train vs test F1 as depth → 30?

In [ ]:
depths = [1, 2, 3, 4, 6, 8, 12, 16, 22, 30]
tr_f1, te_f1 = [], []
for d in depths:
    t = DecisionTreeClassifier(criterion="entropy", max_depth=d, random_state=11).fit(X_train, y_train)
    tr_f1.append(f1_score(y_train, t.predict(X_train))); te_f1.append(f1_score(y_test, t.predict(X_test)))
plt.plot(depths, tr_f1, "o-", label="train F1"); plt.plot(depths, te_f1, "o-", label="test F1")
plt.xlabel("max_depth"); plt.legend(); plt.title("Underfit ← → Overfit")

## 🔨 Does scaling matter for trees? (Week 6 comparison table)

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler().fit(X_train)
t_raw = DecisionTreeClassifier(max_depth=4, random_state=11).fit(X_train, y_train)
t_sc = DecisionTreeClassifier(max_depth=4, random_state=11).fit(sc.transform(X_train), y_train)
print("same predictions:", (t_raw.predict(X_test) == t_sc.predict(sc.transform(X_test))).mean())
pd.Series(t_raw.feature_importances_, index=FEATURES).sort_values().plot.barh(title="Feature importance")

## Why questions
1. Your hand-computed gains: which rule won, and does it match the tree's root split?
2. Why does scaling not change a tree's predictions? (Think: thresholds vs distances.)
3. Where on the depth curve would you pick depth, and why not the depth with the best *train* F1?
4. Try `ccp_alpha=0.001` instead of max_depth. Log both runs and compare in `show_runs("03")`.

---
## 🔧 Git task (do this before you start, and finish after)
```bash
git switch main && git pull
git switch -c module-03-trees          # start of module
# ... work ...
git add notebooks/ experiments/runs.csv
git commit -m "Module 3: entropy by hand, decision tree, overfitting curve"
git push -u origin module-03-trees
```
Then open a **Pull Request** on GitHub → read your own diff → merge → close the module's Issue.

**Git skill this module:** commit in small steps (entropy → tree → depth sweep = 3 commits), not one big commit.